# 01 - Fetch Bike Counts

Fetches raw 15-minute bike-traffic counts for all counting stations from the
[`od-ms/radverkehr-zaehlstellen`](https://github.com/od-ms/radverkehr-zaehlstellen)
GitHub repository, saves them under `data/raw/bike_counts/`, and reports:

- per-station date coverage (first/last timestamp, number of records)
- an **explicit count of missing 15-minute intervals per station** (gaps are
  never silently dropped)

All fetching and schema validation happens in
`src/muenster_bike_forecast/data/bike_counts.py`; this notebook only
orchestrates calls and reports results. Re-running this notebook is safe:
each station's output file is rebuilt deterministically (sorted,
deduplicated), so it never accumulates duplicate rows.

**Note on source data:** the source repo is documented as containing "raw,
uncleaned" data that may have extended gaps (technical failures,
construction). Such gaps are a property of the source, not a bug in this
pipeline - the point of the missing-interval report below is to make them
visible.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make `src/` importable regardless of whether this notebook is run from
# `notebooks/` (the normal case) or the project root.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from muenster_bike_forecast.data.bike_counts import (
    BikeCountDataError,
    fetch_station_data,
    list_stations,
    save_station_data,
    save_stations_index,
    summarize_coverage,
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "bike_counts"
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

## 1. List available stations

Fetches the station index (`site_min.json`) from the source repo: station
id, name, first year of data, and measurement channels.

In [2]:
stations = list_stations()
save_stations_index(stations, RAW_DATA_DIR)

print(f"Found {len(stations)} stations.")
pd.DataFrame(
    [
        {
            "station_id": s.station_id,
            "name": s.name,
            "start_year": s.start_year,
            "n_channels": len(s.channels),
        }
        for s in stations
    ]
)

Found 23 stations.


,station_id,name,start_year,n_channels
0,300038855,Bismarckallee,2023,3
1,300037926,Bohlweg,2023,3
2,300039328,Coesfelder Kreuz,2023,3
3,100034978,Gartenstraße,2023,3
4,300037931,Gasselstiege,2023,7
5,300037925,Goldstraße,2023,3
6,300039331,Grevener Straße,2023,3
7,100031300,Hafenstraße,2020,3
8,100034980,Hammer Straße,2023,3
9,100034982,Hüfferstraße,2023,3


## 2. Fetch and save raw counts for every station

For each station, fetches every available month from its `start_year`
through the current month (missing months, e.g. before start or during a
sensor outage, are skipped - not an error). Each station's full history is
saved as one CSV under `data/raw/bike_counts/<station_id>.csv`.

Fetching is idempotent: `fetch_station_data` deduplicates by timestamp and
sorts before `save_station_data` writes it, so re-running this cell
overwrites each file with the same deterministic content rather than
appending duplicates.

Any station whose data fails schema validation raises `BikeCountDataError`
and stops the notebook here (fail loudly rather than silently skipping bad
data) - per the project's data-engineering conventions, malformed source
content must never be silently dropped.

In [3]:
station_frames: dict[str, pd.DataFrame] = {}

for station in stations:
    print(f"Fetching {station.station_id} ({station.name}) from {station.start_year}...")
    df = fetch_station_data(station)
    if df.empty:
        print(f"  WARNING: no data at all returned for {station.station_id}.")
    else:
        path = save_station_data(df, RAW_DATA_DIR, station.station_id)
        print(f"  {len(df):,} records -> {path}")
    station_frames[station.station_id] = df

Fetching 300038855 (Bismarckallee) from 2023...


  41,660 records -> C:\Users\FloKI\Documents\Daten\muenster-bike-traffic-forecast\data\raw\bike_counts\300038855.csv
Fetching 300037926 (Bohlweg) from 2023...


  87,921 records -> C:\Users\FloKI\Documents\Daten\muenster-bike-traffic-forecast\data\raw\bike_counts\300037926.csv
Fetching 300039328 (Coesfelder Kreuz) from 2023...


  85,658 records -> C:\Users\FloKI\Documents\Daten\muenster-bike-traffic-forecast\data\raw\bike_counts\300039328.csv
Fetching 100034978 (Gartenstraße) from 2023...


  119,196 records -> C:\Users\FloKI\Documents\Daten\muenster-bike-traffic-forecast\data\raw\bike_counts\100034978.csv
Fetching 300037931 (Gasselstiege) from 2023...


...

  55,605 records -> C:\Users\FloKI\Documents\Daten\muenster-bike-traffic-forecast\data\raw\bike_counts\300037932.csv
Fetching 100034983 (Warendorfer Straße) from 2023...


  119,178 records -> C:\Users\FloKI\Documents\Daten\muenster-bike-traffic-forecast\data\raw\bike_counts\100034983.csv
Fetching 300037933 (Weißenburg Str.) from 2023...


  83,075 records -> C:\Users\FloKI\Documents\Daten\muenster-bike-traffic-forecast\data\raw\bike_counts\300037933.csv
Fetching 100034981 (Weseler Straße) from 2023...


  119,212 records -> C:\Users\FloKI\Documents\Daten\muenster-bike-traffic-forecast\data\raw\bike_counts\100034981.csv
Fetching 100020113 (Wolbecker Straße) from 2023...


  120,744 records -> C:\Users\FloKI\Documents\Daten\muenster-bike-traffic-forecast\data\raw\bike_counts\100020113.csv


## 3. Coverage and missing-interval summary

For every station with at least one record, computes date coverage and the
**explicit count of missing 15-minute intervals** within its covered range
(via `summarize_coverage` / `find_missing_intervals` in `bike_counts.py`).
Stations with zero records are reported separately below - not dropped from
the output.

In [4]:
no_data_stations = [
    station_id for station_id, df in station_frames.items() if df.empty
]
coverage_rows = [
    summarize_coverage(df, station_id=station_id)
    for station_id, df in station_frames.items()
    if not df.empty
]

coverage = pd.DataFrame(coverage_rows).drop(columns=["missing_timestamps"])
coverage = coverage.sort_values("n_missing", ascending=False).reset_index(drop=True)

print(f"Stations with data: {len(coverage)} / {len(stations)}")
print(f"Stations with NO data at all: {no_data_stations if no_data_stations else 'none'}")
print(f"Total missing 15-minute intervals across all stations: {coverage['n_missing'].sum():,}")

coverage

Stations with data: 23 / 23
Stations with NO data at all: none
Total missing 15-minute intervals across all stations: 177,594


,station_id,first_timestamp,last_timestamp,n_records,n_expected,n_missing
0,300038855,2024-01-09 00:00:00,2026-07-06 04:45:00,41660,87284,45624
1,300037932,2023-12-01 15:00:00,2026-07-06 01:45:00,55605,90956,35351
2,300037925,2023-12-01 13:30:00,2026-07-05 23:45:00,80729,90954,10225
3,300037933,2023-12-01 15:15:00,2026-07-06 01:45:00,83075,90955,7880
4,100053305,2023-01-01 00:00:00,2026-07-06 01:15:00,116310,123078,6768
5,100031300,2020-01-01 00:00:00,2026-07-06 04:45:00,221612,228308,6696
6,300037544,2023-11-14 13:45:00,2026-07-06 01:45:00,86477,92593,6116
7,300037936,2023-12-01 15:45:00,2026-07-06 02:00:00,85223,90954,5731
8,100031297,2023-01-01 00:00:00,2026-07-06 04:45:00,117463,123092,5629
9,100034982,2023-01-01 00:00:00,2026-07-06 04:45:00,118844,123092,4248


### Missing-interval detail

Saves the full list of missing timestamps per station to
`data/raw/bike_counts/missing_intervals.csv` (one row per missing 15-minute
interval) so gaps are available for downstream inspection, not just
summarized as a count.

In [5]:
_missing_frames = [
    pd.DataFrame(
        {
            "station_id": row["station_id"],
            "missing_timestamp": row["missing_timestamps"],
        }
    )
    for row in coverage_rows
    if row["missing_timestamps"]
]
missing_detail = (
    pd.concat(_missing_frames, ignore_index=True)
    if _missing_frames
    else pd.DataFrame(columns=["station_id", "missing_timestamp"])
)

missing_path = RAW_DATA_DIR / "missing_intervals.csv"
missing_detail.to_csv(missing_path, index=False)
print(f"{len(missing_detail):,} missing intervals written to {missing_path}")

missing_detail.head(20)

177,594 missing intervals written to C:\Users\FloKI\Documents\Daten\muenster-bike-traffic-forecast\data\raw\bike_counts\missing_intervals.csv


,station_id,missing_timestamp
0,300038855,2024-03-31 02:00:00
1,300038855,2024-03-31 02:15:00
2,300038855,2024-03-31 02:30:00
3,300038855,2024-03-31 02:45:00
4,300038855,2024-04-30 05:00:00
5,300038855,2024-04-30 05:15:00
6,300038855,2024-04-30 05:30:00
7,300038855,2024-04-30 05:45:00
8,300038855,2024-04-30 06:00:00
9,300038855,2024-04-30 06:15:00


## Notes / known limitations

- Gaps counted above are **missing 15-minute rows** within each station's
  covered date range. Rows that are *present* but have an empty/NaN count
  value (a different kind of data gap, at the value level rather than the
  row level) are preserved as-is by `parse_station_csv` and are not counted
  here - that is a modeling-time decision, out of scope for this ingestion
  step.
- Timestamps in the source CSVs have no timezone/UTC-offset marker; they are
  treated as naive local time throughout this pipeline (no DST conversion is
  applied). This is worth revisiting once weather data (also time-indexed)
  is joined in.
- Some stations have more than 3 channels (e.g. duplicate/redundant sensor
  channels at a few Kanalpromenade and Promenade stations) - all channels are
  kept as-is; picking a canonical "primary" channel per station is a
  modeling-stage decision, not made here.